# Modern Application Development – I: Comprehensive Lecture Notes  
**Professor Nitin Chandrachoodan, IIT Madras**  
**Screencast by Professor Thejesh G N, IIT Madras**  
**Week 4: Models, Persistent Storage, Relational Databases, SQL & Introduction to Flask**

---

## Table of Contents
1. [Persistent Storage and the Model in MVC](#1-persistent-storage-and-the-model-in-mvc)
2. [Representing Data: Tables, Keys, and Relationships](#2-representing-data-tables-keys-and-relationships)
3. [Mechanisms for Persistent Storage](#3-mechanisms-for-persistent-storage)
4. [From Spreadsheets to Databases](#4-from-spreadsheets-to-databases)
5. [Entity-Relationship (ER) Diagrams](#5-entity-relationship-er-diagrams)
6. [Introduction to SQL](#6-introduction-to-sql)
7. [NoSQL and Alternative Storage Paradigms](#7-nosql-and-alternative-storage-paradigms)
8. [Introduction to Flask – Practical Web Application](#8-introduction-to-flask--practical-web-application)

---

## 1. Persistent Storage and the Model in MVC

### 1.1 The Role of the Model
In the MVC (Model‑View‑Controller) architecture, the **Model** is the component that manages the data, the business logic, and the rules of the application. It is entirely independent of the user interface. While the View handles presentation and the Controller manages the flow, the Model is the single source of truth for the application’s state.

The lecture revisits the **student gradebook** example to anchor the discussion:
- **Students** have a unique ID (roll number), a name, and possibly other attributes (hostel, address).
- **Courses** have a unique course ID and a name.
- **Enrollments / Marks** link a student and a course, optionally recording a score.

This data must be **persistent**: it must survive program restarts, server crashes, and even hardware failures. Without persistence, all data entered by users would vanish the moment the application stops.

### 1.2 Why Persistence Matters
- **Server restarts:** Web servers may be restarted for updates or due to crashes. Data held only in RAM (variables, lists, objects) is lost.
- **Multiple users:** A centralised, persistent store allows all users to see the same consistent data.
- **Long‑term storage:** Academic records need to be kept for years, not minutes.
- **Concurrent access:** Multiple users (e.g., exam section staff) may read/write simultaneously; a database ensures integrity.

Thus, the Model must be backed by some form of **persistent storage** – a way to save data to a non‑volatile medium (disk, SSD, cloud) and retrieve it reliably.

---

## 2. Representing Data: Tables, Keys, and Relationships

### 2.1 The Tabular Data Model
The lecture presents the data as **spreadsheets** because spreadsheets are a familiar, visual way to think about **tabular data**:
- A table consists of **rows** and **columns**.
- Each column has a name (field/attribute), e.g., `StudentID`, `Name`.
- Each row represents one entity (one student, one course).
- The intersection of a row and column is a **cell**, holding a single value.

In the student table:
```
| StudentID | Name    |
|-----------|---------|
| MAD001    | Alice   |
| MAD002    | Bob     |
| MAD003    | Charlie |
```
Similarly, the course table has `CourseID` and `CourseName`.

### 2.2 The Need for Unique Identifiers (Keys)
Names alone are not sufficient to uniquely identify a student because two people can share the same name. Therefore, we introduce a **unique identifier**, or **key**, for each entity:
- For students: a roll number (`MAD001`).
- For courses: a course code (`EE1001`).

A **primary key** is a column (or combination of columns) that uniquely identifies each row in a table. No two rows can have the same primary key value, and it cannot be `NULL`. Using keys eliminates ambiguity and prevents duplication errors.

### 2.3 Representing Relationships
Students and courses have a **many‑to‑many** relationship: a student can enroll in multiple courses, and a course can have many students. To represent this, we create a third table, often called a **junction table** or **associative entity**, e.g., `Enrollments`:

| StudentID | CourseID | Marks |
|-----------|----------|-------|
| MAD001    | BT1010   | 78    |
| MAD001    | AM1100   | 85    |
| MAD002    | EE1001   | 92    |

- The combination `(StudentID, CourseID)` is unique and forms the primary key of this table.
- `StudentID` is a **foreign key** referencing the `Students` table.
- `CourseID` is a foreign key referencing the `Courses` table.

By using keys, we avoid redundancy: the student’s name and course name are stored only once, in their respective tables. This is a fundamental principle of **database normalisation**.

### 2.4 Structured Data and Spreadsheets
Spreadsheets are a natural fit for this kind of data because:
- They inherently display data in rows and columns.
- They can contain multiple sheets (tables) in a single file.
- They allow easy entry, sorting, and filtering.

However, spreadsheets have limitations when it comes to enforcing constraints (primary key uniqueness, foreign key references), handling concurrent access, and performing complex cross‑sheet queries. This motivates the use of dedicated **database management systems (DBMS)**.

---

## 3. Mechanisms for Persistent Storage

Before diving into full‑fledged databases, the lecture considers how one might store application data using only Python.

### 3.1 In‑Memory Data Structures
Using Python’s built‑in data types:
```python
students = ["Alice", "Bob", "Charlie"]
courses = ["Intro to EE", "Calculus", "Applied Mech"]
# Relationships as list of tuples
enrollments = [("Alice", "Intro to EE"), ("Bob", "Calculus"), ("Alice", "Calculus")]
```
This is simple but fragile: it uses names (not IDs), leading to possible duplication and spelling errors. More critically, the data vanishes when the program exits.

### 3.2 Using Keys and Dictionaries
Introducing numeric IDs and dictionaries:
```python
students = {0: "Alice", 1: "Bob", 2: "Charlie"}
courses = {0: "Intro to EE", 1: "Applied Mech", 2: "Calculus"}
enrollments = [(0, 0), (1, 2), (0, 2)]  # (student_id, course_id)
```
This is more robust: we store only IDs in the relationship list, eliminating redundancy and reducing errors. The dictionaries allow fast lookups of names by ID.

### 3.3 Using Classes and Objects
We can encapsulate data and behaviour using Python classes:
```python
class Student:
    next_id = 0   # class variable for auto-incrementing IDs
    def __init__(self, name):
        self.id = Student.next_id
        Student.next_id += 1
        self.name = name
```
- Each object has its own `id` and `name`.
- The class variable `next_id` automatically assigns a unique ID to each new student.
- This is a crude form of **auto‑increment** used in databases.
- Methods can be added to compute properties (e.g., `full_name`), validate data, or perform actions.

**Limitations**: In‑memory objects disappear when the program stops. Moreover, if multiple instances of the program run simultaneously, the class‑variable approach may generate duplicate IDs. Real databases handle this safely under concurrency.

### 3.4 Saving to Disk: Serialisation and File Formats
To achieve persistence, we must write data to a non‑volatile storage medium. Python provides several mechanisms:

1. **Pickling (`pickle` module):** Serialises Python objects to a binary file and restores them exactly. Simple but:
   - Binary format, not human‑readable.
   - Python‑specific; not interoperable with other systems.
   - Security risk if loading untrusted data.

2. **CSV (Comma‑Separated Values) and TSV (Tab‑Separated Values):**
   - Text‑based, human‑readable, and widely supported.
   - Each line is a row; values within a row are separated by commas or tabs.
   - Suitable for simple tabular data (like the three tables of the gradebook).
   - **Drawbacks:**
     - No standard way to represent data types (everything is a string).
     - Handling commas inside values requires escaping/quoting, which varies.
     - No built‑in support for relationships between files.

3. **JSON (JavaScript Object Notation):**
   - Text‑based, hierarchical, and supports basic data types (strings, numbers, booleans, lists, dictionaries).
   - Easily shared between different programming languages.
   - Could store all tables and relationships in a single file, e.g.:
     ```json
     {
       "students": [{"id": 0, "name": "Alice"}, ...],
       "courses": [...],
       "enrollments": [{"student_id": 0, "course_id": 0, "marks": 78}, ...]
     }
     ```
   - Still lacks querying, indexing, and concurrency support.

These file‑based approaches work for small, single‑user applications. For larger, multi‑user, or complex data, we need a **database management system**.

---

## 4. From Spreadsheets to Databases

### 4.1 Shortcomings of Spreadsheets as a Data Store
While spreadsheets visualise tabular data well, they fall short as a robust backend:
- **Limited cross‑referencing:** Performing a lookup across multiple sheets (e.g., “find all students taking a course taught by a specific department”) becomes cumbersome.
- **No native concurrency control:** If multiple users edit the spreadsheet simultaneously, conflicts and data loss can occur.
- **No standardised query language:** Each spreadsheet application has its own formula language; there is no universal SQL.
- **Lack of enforced integrity:** Spreadsheets cannot guarantee that a referenced student ID actually exists in the student sheet (no foreign key enforcement).
- **Scalability:** Spreadsheets are not designed for millions of rows; performance degrades.

### 4.2 Relational Database Management Systems (RDBMS)
A **relational database** organises data into tables (relations) just like our spreadsheets, but with rigorous theoretical foundations (relational algebra). It provides:
- **Schema definition:** You explicitly declare the structure (columns, types, constraints) of each table.
- **Primary keys and foreign keys** with automatic enforcement of referential integrity.
- **Structured Query Language (SQL):** a standard, declarative language for defining, querying, and manipulating data.
- **Transactions (ACID properties):** Atomicity, Consistency, Isolation, Durability – ensuring that a group of operations either fully succeeds or fully fails, leaving the database in a consistent state.
- **Concurrency control:** Multiple users can read/write simultaneously without corrupting data.
- **Indexes:** Data structures that dramatically speed up queries on large tables.
- **Security:** User authentication and fine‑grained access permissions.

The lecture mentions that RDBMSs originated from IBM’s research in the 1970s (System R, the first SQL implementation). The mathematical rigour behind the relational model (E.F. Codd’s work) is what gives it power and reliability.

### 4.3 Structured vs. Semi‑structured Data
- **Structured data** (relational): Fixed schema – every row in a table has exactly the same columns. This enforces uniformity and simplifies query optimisation.
- **Semi‑structured / unstructured data**: Does not follow a rigid schema. Different rows (documents) can have different fields. Examples: JSON documents, XML, key‑value stores.

**NoSQL databases** (MongoDB, CouchDB, Redis) embrace this flexibility. They may be document‑oriented, graph‑based, or key‑value stores. They can scale horizontally with ease and are often used for big data, real‑time analytics, or when the data model evolves rapidly. However, they may sacrifice some consistency guarantees and lack a standard query language (though many now support SQL‑like queries).

The choice between SQL and NoSQL depends on the application’s needs: structure vs. flexibility, consistency vs. availability, and the complexity of queries.

---

## 5. Entity-Relationship (ER) Diagrams

### 5.1 Purpose of ER Diagrams
An **Entity‑Relationship Diagram (ERD)** is a graphical representation of the entities in a system and the relationships among them. It is a design tool used before implementing the database, helping to:
- Clarify the data requirements.
- Communicate the design with stakeholders.
- Detect missing entities or relationships early.
- Serve as a blueprint for creating the actual SQL schema.

The lecture introduces the **Crow’s Foot notation**, a popular style for drawing ERDs, though other notations (Chen, UML, IDEF1X) also exist.

### 5.2 Basic Elements
- **Entity:** Represented as a rectangle with a name (e.g., `Customer`, `Order`, `Shipment`). Each entity maps to a table in the database.
- **Attributes:** Listed inside the entity rectangle. The **primary key** is marked with `PK`, and other fields have their data types and constraints (e.g., `NOT NULL`, `VARCHAR(50)`).
- **Relationship:** A line connecting two entities. The symbols at each end express the cardinality and optionality.

### 5.3 Crow’s Foot Notation in Detail
At each end of the relationship line, two symbols appear:
1. **An inner symbol** indicating **optionality** (participation):
   - A **circle** (`O`): zero – the participation is optional (may be 0).
   - A **vertical line** (`|`): one – the participation is mandatory (exactly 1).
2. **An outer symbol** indicating **cardinality** (how many):
   - A **single line** (`|`): one.
   - A **crow’s foot** (three lines branching out): many.

Thus, combining these, we get the common notations:
- `||---O||---|`? No, let's recall the lecture examples:
  - Exactly one (mandatory one): a vertical line intersecting the relationship line.  
  - Zero or one (optional one): a circle and a vertical line? Usually a circle with a line? Actually in the lecture: "circle indicates zero, and this cross mark indicates many". The exact depiction: a vertical bar for "one", a circle for "zero", and three lines (crow's foot) for "many". They are placed adjacent to the entity. So the combination "zero or one" is a circle and a single bar; "zero or many" is a circle and crow's foot; "exactly one" is two vertical bars? In Crow's foot: the inner segment indicates optionality (circle = optional, bar = mandatory), the outer segment indicates cardinality (bar = one, crow's foot = many). So optional one = circle + bar; mandatory one = bar + bar; optional many = circle + crow's foot; mandatory many = bar + crow's foot.

From the lecture example:
- Customer to Order: Customer side: "may have zero or many orders" – optional many (circle + crow's foot). Order side: "must have exactly one customer" – mandatory one (bar + bar).
- Order to Shipment: Order side: optional many (circle + crow's foot). Shipment side: mandatory one (bar + bar).

### 5.4 Example: E‑Commerce ER Diagram
The lecture walks through a simple e‑commerce model:

```
+----------------+          +----------------+          +----------------+
|   Customer     |          |     Order      |          |   Shipment     |
+----------------+          +----------------+          +----------------+
| PK customer_id |          | PK order_id    |          | PK shipment_id |
|    name        |          |    order_date  |          |    date        |
|                |          | FK customer_id |          | FK order_id    |
+----------------+          +----------------+          +----------------+
      |                              |                            |
      | (0,N)                        | (1,1)                      | (1,1)
      +------------------------------+                            |
                                     | (0,N)                      |
                                     +----------------------------+
```

- **Customer – Order:** A customer can have zero to many orders; an order must belong to exactly one customer.
- **Order – Shipment:** An order can have zero to many shipments (if split); a shipment must be associated with exactly one order.

**Interpreting foreign keys:**
- In the `Order` table, the `customer_id` foreign key enforces the “must have exactly one customer” rule.
- In the `Shipment` table, the `order_id` foreign key enforces the “must have exactly one order” rule.

ERDs make it immediately obvious how tables are related and what constraints apply. Many tools can generate SQL DDL (Data Definition Language) from such diagrams and vice‑versa.

### 5.5 Types of Relationships Summarised
- **One‑to‑one:** e.g., a student and their unique roll number. Usually the roll number can simply be a column in the student table; a separate table is only needed if there is significant extra detail or the roll number is an entity in its own right.
- **One‑to‑many:** e.g., hostel to students. One hostel has many students; each student is in exactly one hostel. Implemented by a foreign key in the “many” side table.
- **Many‑to‑many:** e.g., students and courses. Requires a junction table (associative entity) containing foreign keys to both tables.

The lecture emphasises that these patterns recur constantly and being able to model them correctly is a core skill for application development.

---

## 6. Introduction to SQL

### 6.1 What is SQL?
**SQL (Structured Query Language)** is the standard language for interacting with relational databases. It is declarative: you state *what* you want, not *how* to get it. The DBMS engine figures out the optimal execution plan.

SQL can be divided into several sub‑languages:
- **DDL (Data Definition Language):** `CREATE TABLE`, `ALTER TABLE`, `DROP TABLE`.
- **DML (Data Manipulation Language):** `SELECT`, `INSERT`, `UPDATE`, `DELETE`.
- **DCL (Data Control Language):** `GRANT`, `REVOKE`.

The focus in this lecture is on the `SELECT` statement for querying data.

### 6.2 Basic SELECT Queries
```sql
SELECT column1, column2 FROM table_name WHERE condition;
```
- `SELECT *` returns all columns.
- `WHERE` filters rows based on a condition (`WHERE name = 'Alice'`, `WHERE marks > 80`).

### 6.3 Joins: Combining Tables
The real power of relational databases comes from joining tables using foreign key relationships.

**Inner Join:** Returns only rows where there is a match in both tables.
```sql
SELECT Students.name, Hostels.name
FROM Students
INNER JOIN Hostels ON Students.hostel_id = Hostels.id;
```
This finds the hostel name for each student. Only students who have a valid hostel ID appear; students without a hostel (if NULL allowed) are excluded.

**Cartesian Product (Cross Join):** If no join condition is specified, every row of the first table is paired with every row of the second table (N × M rows). This is rarely useful in itself but forms the mathematical basis for joins. Filters are then applied to narrow down to meaningful combinations.

### 6.4 Multi‑Table Query Example
The lecture poses: “Find all students who are taking Calculus.”

This requires joining three tables (Students, Enrollments, Courses) and filtering by course name.
```sql
SELECT s.name
FROM Students s
JOIN Enrollments sc ON s.id = sc.student_id
JOIN Courses c ON sc.course_id = c.id
WHERE c.name = 'Calculus';
```
- `s`, `sc`, `c` are **aliases** – short nicknames for the table names to make the query more readable.
- The logic flows backwards from the condition: first find the course `Calculus` in `Courses`, get its ID; then find matching rows in `Enrollments`; then fetch the corresponding student names.
- The DBMS optimiser may rearrange the order of operations for efficiency, but the result is the same.

This single query replaces what would have been a multi‑step procedural search in a flat‑file system.

### 6.5 Importance of Understanding SQL for App Development
While many modern frameworks provide Object‑Relational Mappers (ORMs) that generate SQL for you, a solid understanding of the underlying SQL is crucial for:
- Debugging performance issues (slow queries).
- Writing complex reports that are awkward to express in an ORM.
- Understanding what the ORM is doing “under the hood”.
- Interacting with databases directly during development or administration.

The lecture reassures that for this course, heavy SQL knowledge is not strictly required because an ORM (via Flask‑SQLAlchemy) will be used, but a conceptual grasp is essential.

---

## 7. NoSQL and Alternative Storage Paradigms

### 7.1 When Relational Falls Short
Consider extending the student record with highly variable information: a hostel student needs room number and mess preference; a day scholar needs a local address; an exchange student needs foreign university details. Adding a separate column for every possible attribute leads to a table with many `NULL` values (sparse data), which is inefficient and unwieldy.

### 7.2 Document‑Oriented NoSQL
A document database (e.g., MongoDB) stores each student as a self‑describing JSON‑like document:
```json
{
  "student_id": "MAD001",
  "name": "Alice",
  "type": "hostel",
  "room": "A-101",
  "mess": "Veg"
}
```
Another student might have:
```json
{
  "student_id": "MAD002",
  "name": "Bob",
  "type": "exchange",
  "foreign_university": "MIT",
  "exchange_program": "Semester Abroad"
}
```
- No predefined schema is required; each document can have its own set of fields.
- Adding a new attribute for one student does not affect others.
- Queries can still index and search on any field.

### 7.3 Trade‑offs
- **Flexibility** vs **Structure**: NoSQL allows rapid iteration but may lack the discipline that prevents data quality issues.
- **Consistency** vs **Scalability**: Many NoSQL systems relax the ACID guarantees for better horizontal scaling (the CAP theorem). For a banking system, strong consistency is non‑negotiable; for a social media feed, eventual consistency may be acceptable.
- **Query complexity**: Ad‑hoc joins across documents can be less efficient or require denormalisation.

The lecture’s message is not that one is universally better, but that as an app developer, you must choose the right tool for the data characteristics and access patterns.

---

## 8. Introduction to Flask – Practical Web Application

The screencast by Professor Thejesh G N takes the theoretical concepts of the Model and the previously learned Views and Controllers and demonstrates a minimal but complete web application using **Flask**, a lightweight Python web framework.

### 8.1 Setup and Virtual Environment
- Create a project directory and a **Python virtual environment** to isolate dependencies.
  ```bash
  python3 -m venv .experiment-env
  source .experiment-env/bin/activate
  ```
- Create a `requirements.txt` listing `flask` and install with `pip install -r requirements.txt`. This installs Flask and its dependencies (including Jinja2, Werkzeug, MarkupSafe).
- Virtual environments are critical for reproducible, conflict‑free Python projects.

### 8.2 A Minimal Flask Application
```python
from flask import Flask
app = Flask(__name__)

@app.route('/')
def hello():
    return "Hello World!"

if __name__ == '__main__':
    app.run(debug=True)
```
- `Flask(__name__)` creates the application object.
- `@app.route('/')` is a **decorator** that binds the URL path `/` to the `hello()` function.
- The function returns a string, which Flask sends as the HTTP response with a default `text/html` content type.
- `app.run(debug=True)` starts the built‑in development server on `localhost:5000`. Debug mode provides an interactive debugger in the browser when errors occur, and auto‑reloads on code changes.

**Note:** The built‑in server is **not** for production; for deployment, one would use a production WSGI server like Gunicorn or uWSGI behind Nginx.

### 8.3 Routing and Dynamic URLs
- Routes map URL paths to view functions.
- Path parameters can be captured with `<variable_name>`:
  ```python
  @app.route('/user/<username>')
  def show_user(username):
      return f"User: {username}"
  ```
- By default, a route responds only to `GET` requests. To accept other HTTP methods, use the `methods` argument:
  ```python
  @app.route('/submit', methods=['GET', 'POST'])
  ```

### 8.4 Templating with Jinja2
Flask integrates the Jinja2 template engine. By convention, templates are stored in a `templates/` directory.

- Create a template file, e.g., `hello_world.html`:
  ```html
  <!DOCTYPE html>
  <html>
  <body>
    <h1>Hello World! Welcome Back.</h1>
  </body>
  </html>
  ```
- Render it in a view function using `render_template()`:
  ```python
  from flask import render_template
  @app.route('/hello')
  def hello():
      return render_template('hello_world.html')
  ```
- Jinja2 allows dynamic placeholders: `{{ variable }}` for expression substitution, and `{% ... %}` for control flow.

### 8.5 Handling Forms: GET and POST
A common pattern is to serve a form on GET and process its submission on POST.

1. **GET request** – display the form:
   ```python
   @app.route('/hello', methods=['GET', 'POST'])
   def hello():
       if request.method == 'GET':
           return render_template('get_details.html')
       ...
   ```
2. **The form template** (`get_details.html`):
   ```html
   <form method="POST">
     <input type="text" name="username" placeholder="Enter your name">
     <input type="submit" value="Submit">
   </form>
   ```
   - `method="POST"` tells the browser to send the data in the request body, not the URL.
   - The `name` attribute of the input determines the key in the submitted data.
3. **POST request** – process the form data:
   ```python
   if request.method == 'POST':
       username = request.form['username']
       return render_template('display_details.html', display_name=username)
   ```
   - `request.form` is a dictionary‑like object containing parsed form data.
   - The variable `display_name` is passed to the template.
4. **Display template** (`display_details.html`):
   ```html
   <h1>Hello {{ display_name }}!</h1>
   ```

The screencast demonstrates the full round‑trip: browser sends GET, server returns form; user fills name, clicks Submit, browser sends POST, server reads `username`, renders a personalised greeting.

### 8.6 Debugging with Flask
- Setting `debug=True` enables an interactive debugger. If an error occurs (e.g., `NameError: name 'request' is not defined`), the browser shows a traceback with the option to open a Python console right in the stack frame.
- It also auto‑reloads the server when code files change, accelerating development.
- **Important:** Never enable debug mode in production; it can expose code and allow arbitrary code execution.

### 8.7 Separating Logic and Presentation
The screencast shows moving the HTML out of the Python code into template files, reinforcing the MVC separation:
- **Model** (data) is not yet persistent in this screencast – it's just a variable, but the pattern is set.
- **View** (presentation) is the Jinja2 templates.
- **Controller** is the Flask route functions that handle HTTP requests, interact with the model (here, just the request data), and select the appropriate view.

### 8.8 The `request` Object
Flask’s `request` object provides all the information about the incoming HTTP request:
- `request.method` – `GET`, `POST`, etc.
- `request.form` – form data from a POST request.
- `request.args` – query string parameters from a GET request (e.g., `/search?q=python`).
- `request.files` – uploaded files.
- `request.headers` – HTTP headers.

Using `request`, the controller can inspect what the user sent and act accordingly.

### 8.9 Summary of the Flask Exercise
The screencast achieves:
- A basic understanding of routing.
- Use of templates to render HTML dynamically.
- Handling both GET and POST requests on the same endpoint.
- Reading form data and passing it to a template.
- The fundamental request‑response cycle of a web application.

This directly ties into the earlier theoretical discussions: the views are the templates, the controller is the Flask function, and the data (for now, just the username) is the beginning of the model.

---

## Conclusion of Week 4

- **Persistent storage** is the backbone of the Model in MVC. Data must survive beyond program execution.
- Data can be represented using in‑memory structures (lists, dicts, objects) and saved to disk via files (CSV, JSON) or databases.
- **Relational databases** (SQL) offer robust, structured data storage with enforced integrity, powerful querying via SQL, and ACID transactions.
- **Entity‑Relationship diagrams** (Crow’s Foot notation) provide a visual way to design tables and their relationships before coding.
- **SQL** is the standard query language; `JOIN` operations are fundamental for combining data from related tables.
- **NoSQL** databases provide flexible schemas for semi‑structured data, trading some guarantees for scalability.
- **Flask** is a simple yet powerful Python web framework. It uses routes to map URLs to Python functions, Jinja2 for templating, and provides easy handling of HTTP requests and responses.
- The practical exercise demonstrated the core web application pattern: presenting a form, receiving user input, and displaying a personalised response, all while keeping presentation logic separate from application logic.

This week bridges the conceptual model design with hands‑on implementation, setting the stage for building a complete web application with persistent storage in subsequent weeks.